# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# Load Data

In [2]:
df = pd.read_csv('titanic_dataset.csv')

# Over view

### 1. Head

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 2. Column Names

In [4]:
pd.DataFrame(df.columns)

,0
0,PassengerId
1,Survived
2,Pclass
3,Name
4,Sex
5,Age
6,SibSp
7,Parch
8,Ticket
9,Fare


### 3. Column DataTypes

In [5]:
pd.DataFrame(df.dtypes)

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,str
Sex,str
Age,float64
SibSp,int64
Parch,int64
Ticket,str
Fare,float64


### Shape

In [6]:
df.shape

(891, 12)

## conclusion:
- data have **891 Rows** and **12 Columns**.
- we need to Transform / Break following Columns:
    Name : [sur_name, title, Name]<br>
    Fare : [ indivual_fare]<br>
    Sibsp, Parch : [Family size , is_alone]<br>
    Cabin : [Deck, Cabin]
- we need to perform Encoding on:
    1. Sex [binary]
    2. Embarked [one-hot]
    3. Deck [one-hot]

# Transform / Break

### SurName

In [7]:
df['Surname'] = df['Name'].str.split(',').str[0]
df['Surname'] = df['Surname'].astype('category')

### Title

In [8]:
df['Title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
df['Title'] = df["Title"].astype("category")

other_titles = [
    "Countess", "Col", "Don",
    "Major", "Rev", "Jonkheer", "Dona"
]

df["Title"] = df["Title"].str.strip()
df["Title"] = df["Title"].replace(other_titles, "others")

### Family Size

In [9]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

### is Alone

In [10]:
df["is_alone"] = (df["FamilySize"] == 1).astype(int)


### individual Fare per person

In [11]:
df["TicketGroupSize"] = df.groupby("Ticket")["PassengerId"].transform("count")
df["FarePerPerson"] = df["Fare"] / df["TicketGroupSize"]
df['FarePerPerson']

0       7.2500
1      71.2833
2       7.9250
3      26.5500
4       8.0500
        ...   
886    13.0000
887    30.0000
888    11.7250
889    30.0000
890     7.7500
Name: FarePerPerson, Length: 891, dtype: float64

### Cabin

In [12]:
df['Deck'] = df['Cabin'].str[0]
df['Deck']

0      NaN
1        C
2      NaN
3        C
4      NaN
      ... 
886    NaN
887      B
888    NaN
889      C
890    NaN
Name: Deck, Length: 891, dtype: str

In [13]:
df['Age'] = df["Age"].fillna(df["Age"].mean())

In [14]:
df['Age'] = df["Age"].fillna(df["Age"].mean())

# Encoding

### Sex

In [15]:
encoder = LabelEncoder()

df["Sex"] = encoder.fit_transform(df["Sex"])

### Embarked

In [16]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded = encoder.fit_transform(df[["Embarked"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Embarked"]),
    index=df.index
)

df = pd.concat([df, encoded_df], axis=1)

### Deck

In [17]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded = encoder.fit_transform(df[["Deck"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Deck"]),
    index=df.index
)

df = pd.concat([df, encoded_df], axis=1)

# Missing Values

In [18]:
df.isna().sum()

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                  0
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin              687
Embarked             2
Surname              0
Title                0
FamilySize           0
is_alone             0
TicketGroupSize      0
FarePerPerson        0
Deck               687
Embarked_C           0
Embarked_Q           0
Embarked_S           0
Embarked_nan         0
Deck_A               0
Deck_B               0
Deck_C               0
Deck_D               0
Deck_E               0
Deck_F               0
Deck_G               0
Deck_T               0
Deck_nan             0
dtype: int64

# Saving data to new File

In [19]:
df.to_csv('Clean_Titanic_Data.csv', index=False)